In [18]:
import requests
from bs4 import BeautifulSoup
import os
import tempfile
import tarfile
import gzip
import shutil
import subprocess
import glob

In [ ]:
def download(outdir, prefix):
    url = "https://fileshare.lisc.univie.ac.at/vog/latest"

    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Extract all links
    files = []
    for link in soup.find_all('a'):
        href = link.get('href')
        if href and not href.startswith('?') and not href.startswith('/'):
            files.append(href)

    download = list()
    for f in files:
        if (f.startswith("release") or f.startswith("vogdb") or f.startswith(prefix)) and not f.endswith(".md5") and any([filetype in f for filetype in ["annotations", "lca", "members", "virusonly", "faa", "hmm", "msa", "release"]]):
            download.append(f)

    for file in download:
        file_url = url + "/" + file

        print(f"Downloading {file}")

        if file.endswith(".gz"):
            with tempfile.TemporaryDirectory() as tmpdir:
                temp_file = os.path.join(tmpdir, file)
                with requests.get(file_url, stream=True) as r:
                    r.raise_for_status()
                    with open(temp_file, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)

                if file.endswith(".tar.gz"):
                    with tarfile.open(temp_file, 'r:gz') as tar:
                        tar.extractall(path=outdir)
                else:
                    outfile = os.path.join(outdir, file[:-len(".gz")])
                    with gzip.open(temp_file, 'rb') as f_in, open(outfile, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
        
        else:
            outfile = os.path.join(outdir, file)
            with requests.get(file_url, stream=True) as r:
                r.raise_for_status()
                with open(outfile, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)

# Download vogs

In [22]:
outdir = "/p/vast1/ibap/chivian1/proj/viral_fams/dbs/test_VOGDB_vogs_231"
if not os.path.exists(outdir):
    os.makedirs(outdir)

download(outdir, "vog")

In [ ]:
hmm_dir =  os.path.join(outdir, "hmm")
hmm_files = sorted(glob.glob(os.path.join(hmm_dir, "*.hmm")))

output_file = os.path.join(outdir, "vog_combined.hmm")
with open(output_file, "wb") as outfile:
    for fname in hmm_files:
        with open(fname, "rb") as infile:
            outfile.write(infile.read())

subprocess.run(["hmmpress", output_file])

# Download vfams

In [ ]:
outdir = "/p/vast1/ibap/chivian1/proj/viral_fams/dbs/test_VOGDB_vfam_231"
if not os.path.exists(outdir):
    os.makedirs(outdir)

download(outdir, "vfam")

In [21]:
hmm_dir =  os.path.join(outdir, "hmm")
hmm_files = sorted(glob.glob(os.path.join(hmm_dir, "*.hmm")))

output_file = os.path.join(outdir, "vfam_combined.hmm")
with open(output_file, "wb") as outfile:
    for fname in hmm_files:
        with open(fname, "rb") as infile:
            outfile.write(infile.read())

subprocess.run(["hmmpress", output_file])


Error: File format problem in trying to open HMM file /p/vast1/ibap/chivian1/proj/viral_fams/dbs/test_VOGDB_vogs_231/vfam_combined.hmm.
File exists, but appears to be empty?



CompletedProcess(args=['hmmpress', '/p/vast1/ibap/chivian1/proj/viral_fams/dbs/test_VOGDB_vogs_231/vfam_combined.hmm'], returncode=1)